In [ ]:
# 1. 구글 드라이브 연결 (로그인 팝업이 뜨면 확인만 눌러주세요)
from google.colab import drive
drive.mount('/content/drive')

# 2. 구글 드라이브에 저장해둔 열쇠(access_token)를 코랩 보안 폴더로 자동 복사
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/Kaggle/access_token ~/.kaggle/
!chmod 600 ~/.kaggle/access_token

# 3. 캐글 API로 MNIST 데이터 초고속 다운로드 및 압축 해제
!pip install -q kaggle

In [ ]:
!kaggle competitions download -c dog-breed-identification

In [ ]:
!unzip -q dog-breed-identification.zip -d ./dog_breed

!ls -l ./dog_breed

In [ ]:
import numpy as np
import pandas as pd

train_df = pd.read_csv('./dog_breed/labels.csv')
print(len(train_df))
print(train_df.head())

unique_breeds = train_df['breed'].unique()
unique_breeds.sort()

breed_to_idx = {breed: idx for idx, breed in enumerate(unique_breeds)}
idx_to_breed = {idx: breed for idx, breed in enumerate(unique_breeds)}

train_df['label_idx'] = train_df['breed'].map(breed_to_idx)

train_df['img_file'] = train_df['id'] + '.jpg'

print(train_df[['img_file', 'breed', 'label_idx']].head())

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class DogBreedDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['img_file']
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        label = self.df.iloc[idx]['label_idx']

        if self.transform:
            image = self.transform(image)

        return image, label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_img_dir = './dog_breed/train'
full_dataset = DogBreedDataset(df=train_df, img_dir=train_img_dir, transform=train_transform)
print(len(full_dataset))

In [ ]:
from torch.utils.data import DataLoader, random_split

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 120)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(device)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
import time

epochs = 3
for epoch in range(epochs):
    start_time = time.time()

    model.train()
    train_loss, train_correct, train_total = 0.0, 0.0, 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total * 100

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0.0, 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total * 100

    end_time = time.time()
    epoch_mins, epoch_secs = divmod(end_time - start_time, 60)

    print(f"Epoch [{epoch+1} / {epochs}] | Time: {int(epoch_mins)}m {int(epoch_secs)}s")
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}")
    print(f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}\n")


```
images.size(0), labels.size(0) 둘 다 같은 값(배치 사이즈)인데 혼용해서 사용하는 이유?
loss.item() * images.size(0): loss는 보통 입력 이미지 1장당 발생하는 평균 손실값입니다.
따라서 "이번 배치에 들어온 '이미지 개수'만큼 곱해서 전체 손실의 총합을 구하겠다"는 직관적인 계산식으로 image.size(0)을 사용한 것입니다.
train_total += labels.size(0): 정답을 맞힌 개수(train_correct)와 비교할 "전체 정답 레이블의 총개수"를 누적하는 칸이므로 의미상 labels.size(0)을 갖다쓴 것 입니다.
```

In [ ]:
import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

class KaggleTestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        # 폴더 내 모든 jpg, jpeg, png 파일 경로를 리스트로 수집
        self.image_paths = glob.glob(os.path.join(test_dir, '*.jpg')) + \
                            glob.glob(os.path.join(test_dir, '*.jepg')) + \
                            glob.glob(os.path.join(test_dir, '*.png'))
        self.transform = transform

    def __len__(self):
        # 전체 테스트 이미지 개수 반환
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 1. 파일 경로에서 특정 이미지 꺼내기
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')

        # 2. 캐글 제출용 csv에 파일명이 들어가야 하므로 파일명만 추출
        # '/content/test/00123.jpg -> 00123
        image_id = os.path.basename(img_path).split('.')[0]

        if self.transform:
            image = self.transform(image)

        return image, image_id

In [ ]:
# 1. 테스트 전용 전처리 (학습 때 썼던 모델의 입력 크기와 맞춰야 합니다.)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 2. 이미지 폴더 경로 지정
TEST_DIR = './dog_breed/test' # 런타임 연결하고 다시 파일 다운받고 경로 확인 필요

# 3. 데이터셋 인스턴스 생성
test_dataset = KaggleTestDataset(test_dir=TEST_DIR, transform=test_transform)

# shuffle=False 매우 중요. 순서대로 예측해야 순서대로 csv에 담깁니다.
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

print(len(test_dataset))

In [ ]:
images, image_ids = next(iter(test_loader))
print(images.shape)
print(image_ids[:5])

In [ ]:
# 캐글 제출용 csv 생성 실전 코드

# 매우 중요함.. 모델을 평가모드로 전환하는 것은..
model.eval()

all_preds, all_image_ids = [], []

with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(device)

        outputs = model(images)

        _, predicted = outputs.max(1)

        # GPU 메모리에 있는 텐서를 CPU로 내리고, 넘파이 리스트 형태로 변환하여 누적
        all_preds.extend(predicted.cpu().numpy())
        all_image_ids.extend(image_ids)

submission_df = pd.DataFrame({
    'Id': all_image_ids,
    'Category': all_preds
})

submission_df.to_csv('submission.csv', index=False)

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np

model.eval()

all_probs = [] # 모든 이미지의 품종별 확률값(배열)을 담을 리스트
all_image_ids = [] # 이미지 ID들을 담을 리스트

with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(device)

        outputs = model(images)

        # 🔥 핵심: Logits을 0~1 확률값으로 변환
        probs = F.softmax(outputs, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_image_ids.extend(image_ids)

# 분할되어 쌓인 넘파이 배열들을 하나로 병합 (전체테스트개수, 120) 구조가 됩니다.
all_probs = np.vstack(all_probs)

sample_sub = pd.read_csv('./dog_breed/sample_submission.csv')
breed_columns = sample_sub.columns[1:]

# 예측 확률 데이터프레임 생성 (컬럼명을 순서대로 매핑)
submission_df = pd.DataFrame(all_probs, columns=breed_columns)

# 맨 앞에 'id' 열 추가
submission_df.insert(0, 'id', all_image_ids)

# 3. 데이터 검증
print(f"제출 데이터 shape: {submission_df.shape}")
print(submission_df.head(3))

submission_df.to_csv('submission.csv', index=False)

```
all_probs.append(probs.cpu().numpy())
.numpy()의 이유: (Tensor -> NumPy)
파이토치 전용 자료형인 Tensor 상태로는 판다스 데이터프레임으로 깔끔하게 변환하기 어렵습니다.
판다스는 기본적으로 내부 구조가 NumPy 배열을 기반으로 돌아가기 때문에,
NumPy 배열로 변환하는 것이 표준 양식입니다.
append()의 이유: 배치 단위로 나오는 예측 결과물(32개 이미지에 대한 확률 배열 덩어리 하나)을
리스트에 하나의 원소(object)로 툭툭 던져 저장해두기 위함입니다.
all_image_ids.extend(image_ids)
extend()의 이유: 리스트 안에 '알맹이 원소들만' 쏙쏙 꺼내서 이어붙임
append를 쓰면: [['img_1', 'img_2'], ['img_3', 'img_4']]
extend를 쓰면: ['img_1', 'img_2', 'img_3', 'img_4']
```

In [ ]:
# @title
# idx_to_breed를 활용해서 csv를 만드는 코드
import numpy as np
import pandas as pd
import torch

all_preds = []
all_image_ids = []

model.eval()
with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(device)

        # 모델 예측 (logits) 및 확률 변환 (softmax)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)

        # 모델의 예측값 중 가장 높은 확률을 가진 인덱스(숫자) 추출
        # 예: [2, 0, 119..] 같은 Tensor가 나옴
        preds = torch.argmax(probs, dim=1)

        # [중요] GPU 메모리 해제 및 넘파이 변환 후 리스트에 축적
        # 숫자들이 들어있는 1차원 배열이므로 extend를 씁니다.
        all_preds.extend(preds.cpu().numpy())
        all_image_ids.extend(image_ids)

# 2. [핵심] 숫자 인덱스 배열을 문자열 견종 이름 리스트로 변환하기
final_breed_names = [idx_to_breed[idx] for idx in all_preds]

# 3. 판다스 데이터프레임 생성
# submission_df = pd.DataFrame({
#     'Id': all_image_ids,
#     'Category'
# })

In [ ]:
import numpy as np
print(np.isclose(submission_df.iloc[0, 1:].sum(), 1.0))

In [ ]:
import numpy as np
import pandas as pd
import torch

all_probs = []
all_image_ids = []

model.eval()
with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(device)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_image_ids.extend(image_ids)

# 2. [핵심] 배치별로 쪼개져 있는 2차원 배열 리스트를 하나의 큰 2차원 배열로 수직 병합
# 예: (32, 120) 배열 여러 개 -> (10357, 120) 크기의 하나의 넘파이 배열로 변환
final_probs_array = np.vstack(all_probs)

# unique_breeds.sort()가 선행되었으므로 columns에 unique_breeds를 바로 넣어주면 됩니다.
submission_df = pd.DataFrame(final_probs_array, columns=unique_breeds)

submission_df.insert(0, 'id', all_image_ids)

submission_df.to_csv('submission.csv', index=False)